In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

# ==========================================
# [실험 설정 구간] 모든 하이퍼파라미터 통합 
# ==========================================
DATA_PATH = '../CMAPSSData/'
CONFIG = {
    'DATA_ID': 'FD001',
    'RUL_CAP': 125,
    'GAUSS_SIGMA': 2,
    'RANDOM_STATE': 42,
    'TEST_SIZE': 0.2,
    'SCALER_TYPE': 'minmax',  # minmax , Standard
    
    # 사용할 센서 직접 지정 (상수 센서 자동 감지 대신 사용 )
    'BASE_SENSORS': ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 
                     's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21'],
    'SETTING_FEATURES': ['setting_1', 'setting_2', 'setting_3'],

    # 파생변수 설정
    'USE_MA': True,   'WIN_MA': 5,
    'USE_STD': True,  'WIN_STD': 10,  
    'USE_DIFF': True, 'PER_DIFF': 1,
    'USE_EMA': True,  'SPAN_EMA': 10,
    'USE_LAG': True,  'LAG_STEP': 10,
}

# ---------------------------------------------------------
# [1] 데이터 로드 및 초기 전처리 (Capping & 상수 제거)
# ---------------------------------------------------------
def load_and_basic_prep(config):
    # 전체 컬럼 정의
    cols = ['unit_nr', 'time_cycles', 'setting_1', 'setting_2', 'setting_3'] + [f's_{i}' for i in range(1, 22)]
    
    # 데이터 로드
    train = pd.read_csv(f"{DATA_PATH}train_{config['DATA_ID']}.txt", sep='\s+', header=None, names=cols)
    test = pd.read_csv(f"{DATA_PATH}test_{config['DATA_ID']}.txt", sep='\s+', header=None, names=cols)
    y_test_final = pd.read_csv(f"{DATA_PATH}RUL_{config['DATA_ID']}.txt", sep='\s+', header=None, names=['RUL'])['RUL'].values

    # RUL 생성 및 Capping
    train['max_cycle'] = train.groupby('unit_nr')['time_cycles'].transform('max')
    train['RUL'] = (train['max_cycle'] - train['time_cycles']).clip(upper=config['RUL_CAP'])
    train.drop(columns=['max_cycle'], inplace=True)

    # 명시적으로 선택한 컬럼만 유지 (unit_nr, time_cycles, RUL + 선택 센서 + 세팅값 )
    keep_cols = ['unit_nr', 'time_cycles'] + config['SETTING_FEATURES'] + config['BASE_SENSORS']
    
    # Train은 RUL 포함, Test는 RUL 제외하고 선택
    train = train[keep_cols + ['RUL']]
    test = test[keep_cols]
    
    print(f"✅ 센서 선택 완료: {config['BASE_SENSORS']}")
    print(f"✅ 유지된 컬럼 총 {len(train.columns)}개 (RUL 포함)")
    
    return train, test, y_test_final

# 실행 
df_train, df_test, y_test = load_and_basic_prep(CONFIG)

# ---------------------------------------------------------
# [2] 가우시안 스무딩 (Team Style)
# ---------------------------------------------------------
def apply_gaussian_smoothing(df, sigma):
    if sigma <= 0: return df.copy()
    df = df.copy().sort_values(['unit_nr', 'time_cycles']).reset_index(drop=True)
    sensors = [c for c in df.columns if c.startswith('s_')]
    
    for uid in df['unit_nr'].unique():
        mask = df['unit_nr'] == uid
        for col in sensors:
            arr = df.loc[mask, col].to_numpy(dtype=np.float32)
            df.loc[mask, col] = gaussian_filter1d(arr, sigma=sigma, mode='nearest')
    return df

df_train = apply_gaussian_smoothing(df_train, CONFIG['GAUSS_SIGMA'])
df_test = apply_gaussian_smoothing(df_test, CONFIG['GAUSS_SIGMA'])
print(f"✅ 가우시안 스무딩 완료 (Sigma: {CONFIG['GAUSS_SIGMA']})")

# ---------------------------------------------------------
# [3] 데이터 분할 (Train / Val)
# ---------------------------------------------------------
unit_ids = df_train['unit_nr'].unique()
train_units, val_units = train_test_split(unit_ids, test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_STATE'])

train_set = df_train[df_train['unit_nr'].isin(train_units)].copy()
val_set = df_train[df_train['unit_nr'].isin(val_units)].copy()
test_set = df_test.copy()

# ---------------------------------------------------------
# [4] 정규화 (Base Sensors + Settings)
# ---------------------------------------------------------
base_sensors = sorted([c for c in df_train.columns if c.startswith('s_')], key=lambda x: int(x.split('_')[1]))
base_plus_setting = base_sensors + CONFIG['SETTING_FEATURES']

if CONFIG['SCALER_TYPE'] == 'minmax':
    scaler = MinMaxScaler()
else:
    scaler = StandardScaler()

train_set[base_plus_setting] = scaler.fit_transform(train_set[base_plus_setting])
val_set[base_plus_setting] = scaler.transform(val_set[base_plus_setting])
test_set[base_plus_setting] = scaler.transform(test_set[base_plus_setting])
print(f" 정규화 완료 ({CONFIG['SCALER_TYPE']})")

# ---------------------------------------------------------
# [5] 파생변수 생성 (정규화된 데이터 기반)
# ---------------------------------------------------------
def add_features_final(df, features, config):
    df_res = df.copy()
    for col in features:
        group = df_res.groupby('unit_nr')[col]

        if config['USE_MA']:
            df_res[f'{col}_ma'] = group.transform(lambda x: x.rolling(config['WIN_MA'], min_periods=1).mean())
        
        if config['USE_STD']:
            df_res[f'{col}_std'] = group.transform(lambda x: x.rolling(config['WIN_STD'], min_periods=1).std().fillna(0))
            
        if config['USE_DIFF']:
            df_res[f'{col}_diff'] = group.transform(lambda x: x.diff(config['PER_DIFF']).fillna(0))
            
        if config['USE_EMA']:
            df_res[f'{col}_ema'] = group.transform(lambda x: x.ewm(span=config['SPAN_EMA'], adjust=False).mean())
            
        if config['USE_LAG']:
            df_res[f'{col}_lag'] = group.transform(lambda x: x.shift(config['LAG_STEP']).bfill())
                
    return df_res

train_set = add_features_final(train_set, base_sensors, CONFIG)
val_set = add_features_final(val_set, base_sensors, CONFIG)
test_set = add_features_final(test_set, base_sensors, CONFIG)

# 최종 피처 리스트 구성 
X_features_full = base_sensors + CONFIG['SETTING_FEATURES']
if CONFIG['USE_MA']:   X_features_full += [f'{c}_ma' for c in base_sensors]
if CONFIG['USE_STD']:  X_features_full += [f'{c}_std' for c in base_sensors]
if CONFIG['USE_DIFF']: X_features_full += [f'{c}_diff' for c in base_sensors]
if CONFIG['USE_EMA']:  X_features_full += [f'{c}_ema' for c in base_sensors]
if CONFIG['USE_LAG']:  X_features_full += [f'{c}_lag' for c in base_sensors]

# 최종 변수 할당
X_train, y_train = train_set[X_features_full], train_set['RUL']
X_val, y_val = val_set[X_features_full], val_set['RUL']
X_test = test_set.groupby('unit_nr').last()[X_features_full] # 테스트는 마지막 시점만 

print(f"\n 전처리 파이프라인 완료!")
print(f" 최종 피처 수: {len(X_features_full)}개")
print(f" 데이터 형태: X_train{X_train.shape}, X_test{X_test.shape}")
print(f" X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(f" X_train: {y_train.shape}, X_val: {y_val.shape}, X_test: {y_test.shape}")

In [ ]:
# 생성된 모든 피처 리스트 출력 
print(f"총 피처 수: {len(X_train.columns)}개")
print("-" * 30)
for i, col in enumerate(X_train.columns):
    print(f"{i+1:03d}. {col}")

# 데이터 프레임의 상위 5행을 통해 값도 살짝 확인 
# display(X_train.head()) # 주피터 노트북

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

def calculate_nasa_score(y_true, y_pred):
    """C-MAPSS 전용 NASA Score 계산 함수 ㅋ"""
    diff = y_pred - y_true
    # 조기 예측(diff < 0)보다 늦은 예측(diff > 0)에 더 큰 패널티 부여 
    score = np.where(diff < 0, np.exp(-diff/13)-1, np.exp(diff/10)-1)
    return np.sum(score)

def evaluate_model(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    score = calculate_nasa_score(y_true, y_pred)
    print(f"[{model_name}] 평가 결과")
    print(f" RMSE: {rmse:.4f}")
    print(f" NASA Score: {score:.4f}")
    return rmse, score

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import time
import optuna
from optuna.samplers import TPESampler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

def report_performance(y_true, y_pred, model_name):
    """
    예측 결과를 평가하고 시각화하는 공통 함수입니다 
    """
    # 1. 지표 계산 (RMSE)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # 2. NASA Score 계산 (비대칭 페널티)
    # 실제보다 늦게 예측할 때(Overestimation) 더 큰 페널티를 부여하는 방식 
    d = y_pred - y_true
    nasa_score = np.sum(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1))
    
    print(f"\n{'='*55}")
    print(f"📊 [{model_name}] 모델 평가 결과")
    print(f"{'-'*55}")
    print(f"✅ RMSE       : {rmse:.4f}")
    print(f"✅ NASA Score : {nasa_score:.4f}")
    print(f"{'='*55}")
    
    # 3. 트렌드 시각화
    plt.figure(figsize=(15, 5))
    
    # 데이터가 너무 많으면 가독성을 위해 앞부분 일부만 출력 
    display_step = min(100, len(y_true)) 
    
    plt.plot(y_true[:display_step], label='Actual RUL', color='black', linestyle='--', alpha=0.7, marker='o', markersize=4)
    plt.plot(y_pred[:display_step], label=f'Predicted (by {model_name})', color='crimson', linewidth=2, marker='x', markersize=4)
    
    # 오차 범위 시각화 
    plt.fill_between(range(display_step), y_true[:display_step], y_pred[:display_step], color='crimson', alpha=0.1)
    
    plt.title(f'RUL Prediction Trend: {model_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Test Unit Sample Index (Subset)')
    plt.ylabel('Remaining Useful Life (Cycles)')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()
    
    return rmse, nasa_score

In [ ]:
import time
import optuna
from optuna.samplers import TPESampler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

# [사전 설정] 전처리 결과 연결 
# FEATURES_LINEAR와 FEATURES_TREE를 동일하게 X_features_full로 사용하거나 
# 필요시 분리해서 정의하세요!
FEATURES_LINEAR = X_features_full 
FEATURES_TREE = X_features_full

# 하이퍼파라미터
N_TRIALS_LIN = 15
N_TRIALS_TRE = 30 # RF가 무거우면 조절 

# ---------------------------------------------------------
# 4. 선형 모델 학습 (Optuna)
# ---------------------------------------------------------
print(f'=== 4. 선형 모델 학습 (Optuna) ===')
print(f'피처 수: {len(FEATURES_LINEAR)}개 | Trials: {N_TRIALS_LIN}\n')

def get_linear_model(trial, name):
    if name == 'Ridge':
        return Ridge(alpha=trial.suggest_float('alpha', 0.01, 500, log=True), 
                     random_state=CONFIG['RANDOM_STATE'])
    elif name == 'Lasso':
        return Lasso(alpha=trial.suggest_float('alpha', 1e-4, 10, log=True), 
                     max_iter=10000, random_state=CONFIG['RANDOM_STATE'])
    else:  # ElasticNet
        return ElasticNet(
            alpha=trial.suggest_float('alpha', 1e-4, 10, log=True),
            l1_ratio=trial.suggest_float('l1_ratio', 0.05, 0.95),
            max_iter=10000, random_state=CONFIG['RANDOM_STATE'])

linear_models = ['Ridge', 'Lasso', 'ElasticNet']
linear_results = {}

# ---------------------------------------------------------
# 4. 선형 모델 학습 (Optuna) 
# ---------------------------------------------------------
print(f'=== 4. 선형 모델 학습 (Optuna) ===')
linear_results = {}

for mname in linear_models:
    t0 = time.time()
    print(f'\n🚀 [{mname}] Optuna 최적화 중...')

    def objective(trial):
        model = get_linear_model(trial, mname)
        model.fit(X_train[FEATURES_LINEAR], y_train)
        preds = model.predict(X_val[FEATURES_LINEAR])
        return np.sqrt(mean_squared_error(y_val, preds))

    study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=CONFIG['RANDOM_STATE']))
    study.optimize(objective, n_trials=N_TRIALS_LIN)

    # [수정] 최적 파라미터 즉시 출력 
    print(f"✅ {mname} Best Params: {study.best_params}")

    best_p = study.best_params
    best_m = get_linear_model(optuna.trial.FixedTrial(best_p), mname)
    best_m.fit(X_train[FEATURES_LINEAR], y_train)

    rmse, score = report_performance(y_test, best_m.predict(X_test[FEATURES_LINEAR]), f"{mname} (Optuna)")
    linear_results[mname] = {'rmse': rmse, 'score': score, 'params': best_p, 'time': time.time()-t0}
# ---------------------------------------------------------
# 5. 트리 모델 학습 (Optuna)
# ---------------------------------------------------------
print(f'\n=== 5. 트리 모델 학습 (Optuna) ===')
print(f'피처 수: {len(FEATURES_TREE)}개 | Trials: {N_TRIALS_TRE}\n')

def get_tree_model(trial, name):
    if name == 'RandomForest':
        return RandomForestRegressor(
            n_estimators=trial.suggest_int('n_estimators', 100, 300),
            max_depth=trial.suggest_int('max_depth', 5, 30),
            min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
            random_state=CONFIG['RANDOM_STATE'], n_jobs=-1)
    elif name == 'XGBoost':
        return xgb.XGBRegressor(
            n_estimators=trial.suggest_int('n_estimators', 100, 600),
            max_depth=trial.suggest_int('max_depth', 3, 10),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            tree_method='hist', random_state=CONFIG['RANDOM_STATE'], n_jobs=-1, verbosity=0)
    else:  # LightGBM
        return lgb.LGBMRegressor(
            n_estimators=trial.suggest_int('n_estimators', 100, 600),
            max_depth=trial.suggest_int('max_depth', 3, 12),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int('num_leaves', 20, 200),
            random_state=CONFIG['RANDOM_STATE'], n_jobs=-1, verbose=-1)

tree_models = ['RandomForest', 'XGBoost', 'LightGBM']

# ---------------------------------------------------------
# 5. 트리 모델 학습 (Optuna) 
# ---------------------------------------------------------
print(f'\n=== 5. 트리 모델 학습 (Optuna) ===')
tree_results = {}

for mname in tree_models:
    t0 = time.time()
    print(f'\n🚀 [{mname}] Optuna 최적화 중...')

    def objective(trial):
        model = get_tree_model(trial, mname)
        model.fit(X_train[FEATURES_TREE], y_train)
        preds = model.predict(X_val[FEATURES_TREE])
        return np.sqrt(mean_squared_error(y_val, preds))

    study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=CONFIG['RANDOM_STATE']))
    study.optimize(objective, n_trials=N_TRIALS_TRE)

    # [수정] 최적 파라미터 즉시 출력 
    print(f"✅ {mname} Best Params: {study.best_params}")

    best_p = study.best_params
    if mname == 'RandomForest':
        best_m = RandomForestRegressor(**best_p, random_state=CONFIG['RANDOM_STATE'], n_jobs=-1)
    elif mname == 'XGBoost':
        best_m = xgb.XGBRegressor(**best_p, tree_method='hist', random_state=CONFIG['RANDOM_STATE'], n_jobs=-1, verbosity=0)
    else:
        best_m = lgb.LGBMRegressor(**best_p, random_state=CONFIG['RANDOM_STATE'], n_jobs=-1, verbose=-1)
    
    best_m.fit(X_train[FEATURES_TREE], y_train)

    rmse, score = report_performance(y_test, best_m.predict(X_test[FEATURES_TREE]), f"{mname} (Optuna)")
    tree_results[mname] = {'rmse': rmse, 'score': score, 'params': best_p, 'time': time.time()-t0}

# ---------------------------------------------------------
# 6. [추가] 최종 하이퍼파라미터 및 성능 요약 리포트 
# ---------------------------------------------------------
print("\n" + "="*100)
print(f"{'Model Type':<15} | {'Model Name':<15} | {'RMSE':<10} | {'Full Hyperparameters'}")
print("-"*100)

all_results = [('Linear', linear_results), ('Tree', tree_results)]
for category, results in all_results:
    for mname, res in results.items():
        # 파라미터 딕셔너리를 예쁘게 문자열로 변환 
        p_str = str(res['params'])
        print(f"{category:<15} | {mname:<15} | {res['rmse']:<10.4f} | {p_str}")